In [1]:
!nvcc --version
!pip install git+https://github.com/afnan47/cuda.git
%load_ext nvcc_plugin

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
  Cloning https://github.com/afnan47/cuda.git to /tmp/pip-req-build-n9rcmi_i
  Running command git clone --filter=blob:none --quiet https://github.com/afnan47/cuda.git /tmp/pip-req-build-n9rcmi_i
  Resolved https://github.com/afnan47/cuda.git to commit aac710a35f52bb78ab34d2e52517237941399eff
  Preparing metadata (setup.py) ... done
  Created wheel for NVCCPlugin: filename=NVCCPlugin-0.0.2-py3-none-any.whl size=4290 sha256=4e9908e68b388f0e102e73a0e012b145bb165d50d7ee86bee13b67376ed52393
  Stored in directory: /tmp/pip-ephem-wheel-cache-kc6a439p/wheels/e8/cf/c3/c90ca0d0bba7969f9b8670f5624f76d097123d656355c77053
Successfully built NVCCPlugin
created output directory at /content/src
Out bin /content/result.out


In [3]:
%%cu
#include <iostream>
#include <cuda.h>

using namespace std;

#define BLOCK_SIZE 2

/* -------- CUDA KERNEL -------- */
__global__ void gpuMM(float *A, float *B, float *C, int N) {

    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    float sum = 0.0f;

    for (int n = 0; n < N; ++n) {
        sum += A[row * N + n] * B[n * N + col];
    }

    C[row * N + col] = sum;
}

/* -------- MAIN FUNCTION -------- */
int main() {

    int N;
    float K;

    cout << "Enter a value for Size/2 of matrix: ";
    cin >> K;

    K = 1;   // fixed for demo
    N = K * BLOCK_SIZE;

    cout << "\nExecuting Matrix Multiplication\n";
    cout << "Matrix size: " << N << " x " << N << endl;

    /* -------- HOST MEMORY -------- */
    float *hA, *hB, *hC;

    hA = new float[N * N];
    hB = new float[N * N];
    hC = new float[N * N];

    /* -------- INITIALIZE MATRICES -------- */
    for (int i = 0; i < N * N; i++) {
        hA[i] = 2;
        hB[i] = 4;
    }

    /* -------- DEVICE MEMORY -------- */
    float *dA, *dB, *dC;
    int size = N * N * sizeof(float);

    cudaMalloc(&dA, size);
    cudaMalloc(&dB, size);
    cudaMalloc(&dC, size);

    /* -------- COPY DATA TO GPU -------- */
    cudaMemcpy(dA, hA, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dB, hB, size, cudaMemcpyHostToDevice);

    /* -------- THREADS & BLOCKS -------- */
    dim3 threadBlock(BLOCK_SIZE, BLOCK_SIZE);
    dim3 grid(K, K);

    /* -------- PRINT INPUT MATRICES -------- */
    cout << "\nMatrix A:\n";
    for (int row = 0; row < N; row++) {
        for (int col = 0; col < N; col++) {
            cout << hA[row * N + col] << " ";
        }
        cout << endl;
    }

    cout << "\nMatrix B:\n";
    for (int row = 0; row < N; row++) {
        for (int col = 0; col < N; col++) {
            cout << hB[row * N + col] << " ";
        }
        cout << endl;
    }

    /* -------- KERNEL CALL -------- */
    gpuMM<<<grid, threadBlock>>>(dA, dB, dC, N);

    /* -------- COPY RESULT BACK -------- */
    float *C = new float[N * N];
    cudaMemcpy(C, dC, size, cudaMemcpyDeviceToHost);

    /* -------- PRINT RESULT -------- */
    cout << "\nResultant Matrix:\n";
    for (int row = 0; row < N; row++) {
        for (int col = 0; col < N; col++) {
            cout << C[row * N + col] << " ";
        }
        cout << endl;
    }

    cout << "\nFinished.\n";

    /* -------- FREE MEMORY -------- */
    delete[] hA;
    delete[] hB;
    delete[] hC;
    delete[] C;

    cudaFree(dA);
    cudaFree(dB);
    cudaFree(dC);

    return 0;
}

Enter a value for Size/2 of matrix: 
Executing Matrix Multiplication
Matrix size: 2 x 2

Matrix A:
2 2 
2 2 

Matrix B:
4 4 
4 4 

Resultant Matrix:
16 16 
16 16 

Finished.



In [ ]:
# Optimizing tool selection...

# Retrieving Notebook summary.

# Read [](file:///c%3A/Users/ADMIN/OneDrive/Desktop/hpc%20final/cuda_matrix.ipynb#1-1), lines 1 to 10

# The selected code in your notebook (cuda_matrix.ipynb, lines 2-4) consists of three lines that set up the environment for CUDA programming in a Jupyter notebook. These are shell commands and Jupyter magic commands, not standard Python code. I'll explain each line in detail, including what it does, why it's needed, and any potential considerations.

# ### Line 1: `!nvcc --version`
# - **What it does**: This is a shell command (prefixed with `!` to run it in the system shell from within Jupyter) that checks and displays the version of `nvcc`, which is the NVIDIA CUDA Compiler (nvcc). It queries the installed CUDA toolkit and prints details like the version number, build date, and copyright information.
# - **Why it's needed**: CUDA programming requires the NVIDIA CUDA toolkit to compile and run GPU-accelerated code. This command verifies that CUDA is installed and accessible. If CUDA isn't installed or the path isn't set up correctly, this will fail (e.g., "nvcc: command not found"). It's a common first step in CUDA setups to ensure compatibility.
# - **Output example**: Something like `nvcc: NVIDIA (R) Cuda compiler driver Copyright (c) 2005-2023 NVIDIA Corporation Built on Wed_Nov_22_10:17:15_PST_2023 Cuda compilation tools, release 12.3, V12.3.107 Build cuda_12.3.r12.3/compiler.33567101_0`.
# - **Potential issues**: If you're on a system without NVIDIA GPUs or CUDA drivers, this might not work. On Windows, ensure CUDA is installed via the NVIDIA installer and that the PATH environment variable includes the CUDA bin directory (e.g., `C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.3\bin`).
# - **Execution context**: This runs in the terminal/shell, not in Python. In Jupyter, the `!` prefix executes it synchronously.

# ### Line 2: `!pip install git+https://github.com/afnan47/cuda.git`
# - **What it does**: This is another shell command that uses `pip` (Python's package installer) to install a Python package directly from a GitHub repository. It clones the repo at `https://github.com/afnan47/cuda.git` and installs it as a package. The `git+` prefix tells pip to treat it as a Git repository source.
# - **Why it's needed**: The repository likely contains a custom CUDA-related Python package or extension (possibly a wrapper or plugin for CUDA in Python/Jupyter). This is often used for tools like `nvcc_plugin` (see line 3), which allows running CUDA code in Jupyter cells. Without this, the extension in line 3 won't load.
# - **Details on the repo**: `afnan47/cuda` appears to be a fork or custom repo providing CUDA utilities. It might include Python bindings or Jupyter integrations for CUDA. Installing from Git means you're getting the latest (or specific) version, but it could be unstable compared to PyPI packages.
# - **Potential issues**: 
#   - Requires Git to be installed (for cloning the repo).
#   - Might fail if the repo is private, doesn't exist, or has dependencies (e.g., NumPy, CUDA runtime). Check for error messages like "Could not find a version that satisfies the requirement."
#   - On Windows, ensure pip is up-to-date (`pip install --upgrade pip`) and that you have permissions to install packages.
#   - This installs globally or in the current environment; if using virtual environments, activate it first.
# - **Execution context**: Runs in the shell via pip. It may take time to download and install, and you'll see progress output.

# ### Line 3: `%load_ext nvcc_plugin`
# - **What it does**: This is a Jupyter "magic command" (prefixed with `%`) that loads a Jupyter extension called `nvcc_plugin`. Extensions add functionality to Jupyter, and this one specifically enables compiling and running CUDA code directly in notebook cells using the `%%cu` magic (as seen in your next cell).
# - **Why it's needed**: Standard Jupyter doesn't support CUDA compilation. This extension integrates `nvcc` (from line 1) into Jupyter, allowing you to write CUDA kernels in cells and compile them on-the-fly. Without it, CUDA code cells (like `%%cu`) won't work.
# - **How it works**: After loading, you can use `%%cu` at the start of a cell to indicate CUDA code. The extension compiles it with nvcc and runs it on the GPU. It likely depends on the package installed in line 2.
# - **Potential issues**: 
#   - If the extension isn't installed (from line 2), this will fail with "No module named 'nvcc_plugin'" or similar.
#   - Requires a compatible CUDA setup (GPUs, drivers). On non-NVIDIA systems, it won't function.
#   - If Jupyter is running in an environment without the extension, restart the kernel after installation.
# - **Execution context**: This is a Jupyter-specific command that modifies the notebook's runtime environment. It loads once per session.

# ### Overall Context and Tips
# - **Purpose**: These lines set up a CUDA development environment in Jupyter, allowing you to write, compile, and run GPU-accelerated matrix operations (based on the notebook name cuda_matrix.ipynb).
# - **Order matters**: They must run in sequence—check CUDA first, install dependencies, then load the extension.
# - **Running the cell**: Since none of the cells have been executed yet, run this cell first. If errors occur, address them (e.g., install CUDA if missing).
# - **Security/Compatibility**: Installing from GitHub can introduce untrusted code—review the repo if possible. Ensure your system has NVIDIA drivers and CUDA toolkit installed.
# - **Next steps**: After this, your second cell uses `%%cu` to define CUDA code, which this setup enables.

# If you meant a different selection or need clarification on any part, let me know!

# Read [](file:///c%3A/Users/ADMIN/OneDrive/Desktop/hpc%20final/cuda_matrix.ipynb#120-120), lines 120 to 124

# Read [](file:///c%3A/Users/ADMIN/OneDrive/Desktop/hpc%20final/cuda_matrix.ipynb#7-7), lines 7 to 120

# Below is a detailed, line-by-line explanation of the entire code in your notebook's second cell (lines 7–120). This is a complete CUDA C++ program for performing matrix multiplication on a GPU. I'll break it down sequentially, explaining what each line or block does, its purpose, and any key concepts. The code uses CUDA for GPU acceleration, so I'll highlight CUDA-specific elements.

# The program multiplies two square matrices (A and B) to produce C, where all elements are initialized to simple values (2 and 4) for demonstration. It runs on the GPU using a kernel function.

# ### Line 7: `%%cu`
# - **What it does**: This is a Jupyter magic command indicating that the following code is CUDA code (C++ with CUDA extensions). It tells the `nvcc_plugin` (loaded in the first cell) to compile and run this as CUDA code on the GPU.
# - **Why it's needed**: Standard Python/Jupyter can't compile CUDA. This magic enables CUDA compilation within the notebook.
# - **Details**: Everything after this line is treated as C++ code until the cell ends.

# ### Lines 8–9: `#include <iostream>` and `#include <cuda.h>`
# - **What they do**: These are preprocessor directives that include header files.
#   - `<iostream>`: Provides input/output streams (e.g., `cout`, `cin`) for console interaction.
#   - `<cuda.h>`: Includes CUDA runtime API headers for GPU functions like `cudaMalloc` and kernel launches.
# - **Why needed**: `<iostream>` for printing matrices and user input. `<cuda.h>` for all CUDA operations.
# - **Details**: These are standard includes in CUDA programs.

# ### Line 11: `using namespace std;`
# - **What it does**: Brings the `std` namespace into scope, allowing shorthand use of standard library elements (e.g., `cout` instead of `std::cout`).
# - **Why needed**: Simplifies code by avoiding `std::` prefixes.
# - **Details**: Common in C++ but can lead to name conflicts; here it's fine for a simple program.

# ### Line 13: `#define BLOCK_SIZE 2`
# - **What it does**: Defines a preprocessor macro `BLOCK_SIZE` as 2. This sets the size of thread blocks in the GPU grid.
# - **Why needed**: Controls how threads are organized in CUDA. A block size of 2x2 means 4 threads per block.
# - **Details**: Used later in `dim3 threadBlock(BLOCK_SIZE, BLOCK_SIZE)`. This is a small value for demo; real apps use 16x16 or 32x32 for efficiency.

# ### Lines 15–16: `/* -------- CUDA KERNEL -------- */` and `__global__ void gpuMM(float *A, float *B, float *C, int N) {`
# - **What they do**: 
#   - The comment labels the kernel function.
#   - `__global__` declares `gpuMM` as a CUDA kernel (a function that runs on the GPU, callable from the host/CPU).
#   - Parameters: `A`, `B`, `C` are pointers to float arrays (matrices), `N` is the matrix size.
# - **Why needed**: Kernels perform parallel computations on the GPU. This kernel computes one element of the result matrix C.
# - **Details**: `__global__` means it's launched from CPU but executes on GPU. `void` return type is standard for kernels.

# ### Lines 17–18: `int row = blockIdx.y * blockDim.y + threadIdx.y;` and `int col = blockIdx.x * blockDim.x + threadIdx.x;`
# - **What they do**: Calculate the row and column indices for the current thread.
#   - `blockIdx.y/x`: Block index in the grid (y for rows, x for columns).
#   - `blockDim.y/x`: Threads per block in y/x direction.
#   - `threadIdx.y/x`: Thread index within the block.
# - **Why needed**: Each thread computes one element of C. This maps the thread's position to a matrix element.
# - **Details**: For a 2D grid, this ensures each thread handles a unique (row, col) in the matrix.

# ### Line 20: `float sum = 0.0f;`
# - **What it does**: Initializes a local variable `sum` to 0.0 (float type, with `f` suffix for clarity).
# - **Why needed**: Accumulates the dot product for the matrix element this thread is computing.
# - **Details**: Local to each thread; no shared memory here.

# ### Lines 22–25: `for (int n = 0; n < N; ++n) {` to `C[row * N + col] = sum;`
# - **What they do**: 
#   - The loop iterates over `n` from 0 to N-1, computing the dot product: `sum += A[row * N + n] * B[n * N + col];`
#   - After the loop, assigns `sum` to `C[row * N + col]`.
# - **Why needed**: Performs the matrix multiplication: C[row][col] = sum over n of A[row][n] * B[n][col].
# - **Details**: Row-major order access (e.g., `A[row * N + n]`). This is the core computation, done in parallel across threads.

# ### Line 26: `}`
# - **What it does**: Closes the kernel function.
# - **Details**: End of `gpuMM`.

# ### Lines 28–29: `/* -------- MAIN FUNCTION -------- */` and `int main() {`
# - **What they do**: Comment and start of the `main` function (program entry point).
# - **Why needed**: `main` runs on the CPU and orchestrates the GPU work.
# - **Details**: Standard C++ entry point.

# ### Lines 31–32: `int N;` and `float K;`
# - **What they do**: Declare variables: `N` for matrix size, `K` for user input (later fixed).
# - **Why needed**: `N` is the actual size; `K` is a multiplier.
# - **Details**: `N = K * BLOCK_SIZE` later.

# ### Lines 34–35: `cout << "Enter a value for Size/2 of matrix: ";` and `cin >> K;`
# - **What they do**: Prompts user for input and reads it into `K`.
# - **Why needed**: Allows dynamic sizing, but overridden next.
# - **Details**: Uses `iostream`. In a notebook, input might not work interactively.

# ### Line 37: `K = 1;   // fixed for demo`
# - **What it does**: Overrides user input, setting `K` to 1.
# - **Why needed**: Fixes the size for demonstration (avoids large matrices).
# - **Details**: `N = 1 * 2 = 2`, so 2x2 matrices.

# ### Lines 39–41: `cout << "\nExecuting Matrix Multiplication\n";` and `cout << "Matrix size: " << N << " x " << N << endl;`
# - **What they do**: Prints status messages.
# - **Why needed**: User feedback.
# - **Details**: Uses `cout` for output.

# ### Lines 43–46: `/* -------- HOST MEMORY -------- */` to `hC = new float[N * N];`
# - **What they do**: Allocates host (CPU) memory for matrices A, B, C using `new`.
# - **Why needed**: Stores matrices on CPU before/after GPU computation.
# - **Details**: Dynamic allocation; `N*N` floats each.

# ### Lines 48–52: `/* -------- INITIALIZE MATRICES -------- */` to `}`
# - **What they do**: Loops to set all elements of A to 2 and B to 4.
# - **Why needed**: Simple initialization for demo (no random values).
# - **Details**: `for (int i = 0; i < N * N; i++)` iterates over all elements.

# ### Lines 54–57: `/* -------- DEVICE MEMORY -------- */` to `cudaMalloc(&dC, size);`
# - **What they do**: Allocates GPU memory for A, B, C using `cudaMalloc`.
# - **Why needed**: GPU needs its own memory space.
# - **Details**: `size = N * N * sizeof(float)` calculates bytes. `&dA` passes pointer by reference.

# ### Lines 59–61: `/* -------- COPY DATA TO GPU -------- */` to `cudaMemcpy(dB, hB, size, cudaMemcpyHostToDevice);`
# - **What they do**: Copies host data to device using `cudaMemcpy`.
# - **Why needed**: Transfers input matrices to GPU.
# - **Details**: `cudaMemcpyHostToDevice` specifies direction.

# ### Lines 63–64: `/* -------- THREADS & BLOCKS -------- */` and `dim3 threadBlock(BLOCK_SIZE, BLOCK_SIZE);` and `dim3 grid(K, K);`
# - **What they do**: Defines thread organization.
#   - `threadBlock`: 2x2 threads per block.
#   - `grid`: KxK blocks (here 1x1, since K=1).
# - **Why needed**: CUDA requires specifying grid/block dimensions for kernel launch.
# - **Details**: `dim3` is a CUDA struct for 3D dimensions (z=1 here).

# ### Lines 66–77: `/* -------- PRINT INPUT MATRICES -------- */` to nested loops printing A and B
# - **What they do**: Prints matrices A and B row by row.
# - **Why needed**: Shows input data.
# - **Details**: Nested loops: outer for rows, inner for columns. Uses `cout`.

# ### Line 79: `/* -------- KERNEL CALL -------- */` and `gpuMM<<<grid, threadBlock>>>(dA, dB, dC, N);`
# - **What they do**: Launches the kernel on GPU.
# - **Why needed**: Executes the parallel computation.
# - **Details**: `<<<grid, threadBlock>>>` syntax launches with specified dimensions. Passes device pointers.

# ### Lines 81–83: `/* -------- COPY RESULT BACK -------- */` to `cudaMemcpy(C, dC, size, cudaMemcpyDeviceToHost);`
# - **What they do**: Copies result from GPU to host.
# - **Why needed**: Retrieves computed matrix C.
# - **Details**: `cudaMemcpyDeviceToHost` for GPU-to-CPU transfer. Note: `C` is a new host array.

# ### Lines 85–96: `/* -------- PRINT RESULT -------- */` to printing C
# - **What they do**: Prints the result matrix C.
# - **Why needed**: Displays output.
# - **Details**: Same nested loop structure as before.

# ### Lines 98–99: `cout << "\nFinished.\n";`
# - **What they do**: Prints completion message.
# - **Details**: Simple output.

# ## Lines 101–106: `/* -------- FREE MEMORY -------- */` to `cudaFree(dC);`
# - **What they do**: Frees allocated memory.
#   - `delete[]` for host arrays.
#   - `cudaFree` for device arrays.
# - **Why needed**: Prevents memory leaks.
# - **Details**: Essential in CUDA programs.

# ### Lines 108–109: `return 0;` and `}`
# - **What they do**: Returns 0 (success) and ends `main`.
# - **Details**: Standard program exit.

# ### Overall Notes
# - **Execution Flow**: CPU allocates/init/copies to GPU → GPU kernel computes → CPU copies back/prints/frees.
# - **CUDA Concepts**: Kernels run in parallel; each thread computes one C element.
# - **Performance**: Small matrices (2x2) here; real apps handle thousands. No error checking (e.g., for `cudaMalloc` failures).
# - **Potential Issues**: Requires CUDA-compatible GPU. In notebook, ensure first cell ran successfully.
# - **Output**: For N=2, A and B are 2x2 matrices of 2s and 4s; C will be 2x2 of 16s (since 2*4*2 + 2*4*2 = 16).

# If you run this and encounter issues, or need help with a specific part, let me know!